In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph,START,END
from loguru import logger
from rich import print


#1. サブグラフを構築
#1.1 サブグラフの状態を宣言
class SubgraphState(TypedDict):
    raw_text:str #未加工のテキスト
    stripped_text:str # 前後の空白を除去
    punctuated_text:str # 文末に句点を追加

#1.2 サブグラフのノードを宣言 -> 空白を除去するノード
def subgraph_strip_node(state:SubgraphState) -> SubgraphState:
    raw_text = state["raw_text"]
    stripped_text = raw_text.strip()
    logger.info("サブグラフ内の strip ノードを呼び出し")
    return {
        "stripped_text":stripped_text
    }

#1.3 句点を追加するノード
def subgraph_punctuated_node(state:SubgraphState) -> SubgraphState:
    stripped_text = state["stripped_text"]
    punctuated_text = stripped_text + '。'
    logger.info("サブグラフ内の punctuated ノードを呼び出し")
    return {
        "punctuated_text":punctuated_text
    }

#1.4 サブグラフを構築
builder = StateGraph(state_schema=SubgraphState)
builder.add_node("subgraph_strip_node",subgraph_strip_node)
builder.add_node("subgraph_punctuated_node",subgraph_punctuated_node)
builder.add_edge(START,"subgraph_strip_node")
builder.add_edge("subgraph_strip_node","subgraph_punctuated_node")
builder.add_edge("subgraph_punctuated_node",END)
subgraph = builder.compile()

res = subgraph.invoke({
    "raw_text":"   langgraphは本当に面白い   "
})
print(res)



In [ ]:
#2.親グラフを構築
#2.1 状態を宣言
class ParentState(TypedDict):
    input_text:str
    cleaned_text:str

#2.2 ノードを宣言 => サブグラフノードを呼び出す
def call_subgraph(state:ParentState)->ParentState:
    input_text = state["input_text"]
    res = subgraph.invoke({"raw_text":input_text})
    cleaned_text = res["punctuated_text"]
    return {
        "cleaned_text":cleaned_text
    }

#2.3 親グラフを構築
parent_builder = StateGraph(state_schema=ParentState)

parent_builder.add_node("call_subgraph",call_subgraph)

parent_builder.add_edge(START,"call_subgraph")
parent_builder.add_edge("call_subgraph",END)

parent_graph = parent_builder.compile()

#2.4 親グラフを呼び出す => 内部関数がサブグラフを呼び出す
res = parent_graph.invoke({"input_text":"    langgraphは本当に楽しい   "})
print(res)

In [ ]:
from IPython.display import display,Image
display(
    Image(parent_graph.get_graph(xray=True).draw_mermaid_png())
)

In [ ]:
list(parent_graph.get_subgraphs())